In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import statsmodels.api as sm
import seaborn as sns
import bambi as bmb
import arviz as az

def remove_nulls_simple(data_subset, variables):
        # Drop rows where any of the specified variable columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=variables, ignore_index=True)
        # Subsets the cleaned dataframe back into only chosen variables
        vars_cleaned = data_subset_cleaned[variables]
        return vars_cleaned

def list_to_str(list_name):
    string = " + ".join(list_name)
    return string

pa_gs = pd.read_csv('data/pa_data.csv')
temps = pd.read_csv("data/pa_temps_supplement.csv")
#combining main csv dataset with supplementary temperature data csv
pa_gs = pa_gs.merge(temps, on=["station_id", "year"], how="left")

#limiting dataset to just time interval of interest
pa_gs = pa_gs[pa_gs['year'] >= 1960]
#dropping non-numeric columns
pa_gs = pa_gs.drop(['last_spring_frost_date', 'first_fall_frost_date', 'station_name','state'], axis=1)

In [5]:
stn_rdm_effect_model = bmb.Model("growing_season_length ~ year + (1|station_id)", pa_gs, dropna = True)
stn_rdm_effect_output = stn_rdm_effect_model.fit()

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.205       15          1109.63 draws/s      0:00:01    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.189       15          1046.82 draws/s      0:00:01    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.107       15          1288.03 draws/s      0:00:01    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.220       15          1336.94 draws/s      0:00:01    0:00:00

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 3 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [6]:
az.summary(stn_rdm_effect_output)

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
sigma,18.24,0.162,18,18,4483,3100,1.00,0.0024,0.0017
Intercept,-592,29.8,-640,-540,2804,2813,1.00,0.56,0.39
year,0.3801,0.015,0.36,0.4,2794,2831,1.00,0.00028,0.0002
1|station_id_sigma,24.6,1.3,23,27,196,345,1.02,0.094,0.062
1|station_id[USC00360022],9.5,5.1,1.2,18,805,2071,1.01,0.18,0.12
...,...,...,...,...,...,...,...,...,...
1|station_id[USW00014860],18.7,2.8,14,23,257,614,1.02,0.17,0.12
1|station_id[USW00014861],38.5,4.4,32,46,690,1611,1.01,0.17,0.12
1|station_id[USW00093778],-2.5,3.8,-8.5,3.5,545,1198,1.01,0.16,0.12
1|station_id[USW00094732],37.3,2.8,33,42,239,655,1.02,0.18,0.13


In [7]:
cov_all = list_to_str(['dtr_annual','tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station', 'dtr_spring',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us'])

formula_no_sigma = bmb.Formula(f"growing_season_length ~ {cov_all} + year")
model_no_sigma = bmb.Model(formula_no_sigma, data=pa_gs, dropna = True)
output_no_sigma = model_no_sigma.fit()

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.128       31          325.37 draws/s       0:00:06    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.259       31          318.00 draws/s       0:00:06    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.235       31          310.97 draws/s       0:00:06    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.194       31          320.68 draws/s       0:00:06    0:00:00

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 10 seconds.


In [8]:
az.summary(output_no_sigma)

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
sigma,17.365,0.153,17,18,6197,3309,1.00,0.0019,0.0014
Intercept,52,69,-58,160,3082,3166,1.00,1.2,0.86
dtr_annual,-10.12,0.44,-11,-9.4,3020,2820,1.00,0.008,0.0058
tmean_spring,1.157,0.268,0.73,1.6,3538,3022,1.00,0.0045,0.0033
tmean_fall,5.087,0.266,4.7,5.5,4090,3325,1.00,0.0042,0.0029
latitude,-2.17,0.59,-3.1,-1.2,4138,3124,1.00,0.0091,0.0068
longitude,1.437,0.21,1.1,1.8,3448,2872,1.00,0.0036,0.0026
tmax_annual,4.51,0.45,3.8,5.2,2951,2892,1.00,0.0083,0.0058
oni_annual,0.02,0.492,-0.78,0.78,5360,2957,1.00,0.0067,0.0047
nao_annual,-0.41,0.83,-1.7,0.89,4004,3252,1.00,0.013,0.0094
